# Day 2 — Re-evaluate the new checkpoints (Kaggle T4)

Run this AFTER `kaggle_train_illum.ipynb` finishes and you've uploaded the resulting `.pth` files as a Kaggle dataset.

Produces:
1. **Headline table for the +illum model** — eval15 + LOL-v2 Real, 5 + 20 DDIM steps, with LPIPS.
2. **Method ablation row** — baseline (no_prior, retrained on LOL-v2 Real) vs +illum.
3. **Step ablation for +illum** — 5/10/20/50/100 DDIM steps to confirm the few-step convergence story holds for the new model.
4. **Adaptive Residual Rescaling (ARR) grid** on the +illum model — your secondary contribution stacked on top.
5. **T4 efficiency** for the new model.
6. **Updated Figure 1 (teaser) and Figure 3 (step ablation)** rendered from the new outputs.

Wall time: ~45–75 min total.

In [ ]:
# === Cell 1: install ===
!pip install -q --upgrade pip
!pip install -q scikit-image lpips fvcore matplotlib

In [ ]:
# === Cell 2: clone repo ===
import os, sys, shutil, subprocess
BRANCH = "main"
REPO_URL = "https://github.com/chirana07/Diffusion_new_final.git"
REPO_DIR = "/kaggle/working/Diffunet"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
# === Cell 3: discover datasets and the new checkpoints ===
import glob

def find_one(patterns, label):
    hits = sorted({h for pat in patterns for h in glob.glob(pat, recursive=True)})
    if not hits:
        print(f"  [{label}] NOT FOUND. Searched: {patterns}")
        return None
    print(f"  [{label}] = {hits[0]}")
    return hits[0]

EVAL15_LOW  = find_one(["/kaggle/input/**/eval15/low", "/kaggle/input/**/eval15/Low"],   "EVAL15_LOW")
EVAL15_HIGH = find_one(["/kaggle/input/**/eval15/high", "/kaggle/input/**/eval15/High"], "EVAL15_HIGH")
LOLV2_LOW   = find_one(["/kaggle/input/**/Real_captured/Test/Low"],   "LOLV2_LOW")
LOLV2_HIGH  = find_one(["/kaggle/input/**/Real_captured/Test/Normal"], "LOLV2_HIGH")

ILLUM_CKPT    = find_one(["/kaggle/input/**/best_lolv2real_illum.pth"],    "ILLUM_CKPT")
BASELINE_CKPT = find_one(["/kaggle/input/**/best_lolv2real_baseline.pth"], "BASELINE_CKPT")

# === MANUAL OVERRIDE if needed ===
# ILLUM_CKPT    = "/kaggle/input/lumidiff-day2-ckpts/best_lolv2real_illum.pth"
# BASELINE_CKPT = "/kaggle/input/lumidiff-day2-ckpts/best_lolv2real_baseline.pth"

missing = [n for n, v in [("EVAL15_LOW", EVAL15_LOW), ("EVAL15_HIGH", EVAL15_HIGH),
                            ("LOLV2_LOW", LOLV2_LOW), ("LOLV2_HIGH", LOLV2_HIGH),
                            ("ILLUM_CKPT", ILLUM_CKPT)] if not v]
if missing:
    raise SystemExit(f"Missing required: {missing}. ILLUM_CKPT is mandatory; BASELINE_CKPT optional.")

if BASELINE_CKPT is None:
    print("\n[note] Baseline checkpoint not found — method-ablation row will be skipped. "
          "That's OK if you only had time for one training run.")

EVAL_RESULTS = "/kaggle/working/eval_results_day2"
os.makedirs(EVAL_RESULTS, exist_ok=True)

SPLITS = [
    f"eval15:{EVAL15_LOW}:{EVAL15_HIGH}",
    f"lolv2_real:{LOLV2_LOW}:{LOLV2_HIGH}",
]

In [ ]:
# === Cell 4: GPU sanity ===
import torch
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

In [ ]:
# === Cell 5: HEADLINE eval — +illum at 5 and 20 steps with LPIPS ===
import csv, sys, subprocess
for STEPS in (5, 20):
    cmd = [
        sys.executable, "evaluation.py",
        "--splits", *SPLITS,
        "--inference-steps", str(STEPS),
        "--sampler", "ddim",
        "--checkpoint", ILLUM_CKPT,
        "--results-root", os.path.join(EVAL_RESULTS, "headline_illum"),
        "--tag", f"s{STEPS}",
    ]
    print("$ " + " ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
# === Cell 6: METHOD ABLATION — baseline (no_prior, lolv2real) vs +illum (lolv2real) at 5 steps ===
if BASELINE_CKPT is not None:
    cmd = [
        sys.executable, "evaluation.py",
        "--splits", *SPLITS,
        "--inference-steps", "5",
        "--sampler", "ddim",
        "--checkpoint", BASELINE_CKPT,
        "--results-root", os.path.join(EVAL_RESULTS, "method_ablation"),
        "--tag", "baseline_s5",
    ]
    print("$ " + " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Skipping method ablation (no baseline checkpoint).")

In [ ]:
# === Cell 7: STEP ABLATION on the +illum model ===
# Confirms the 'few-step convergence' claim survives the architecture change.
for N in (5, 10, 20, 50, 100):
    cmd = [
        sys.executable, "evaluation.py",
        "--splits", *SPLITS,
        "--inference-steps", str(N),
        "--sampler", "ddim",
        "--checkpoint", ILLUM_CKPT,
        "--results-root", os.path.join(EVAL_RESULTS, "step_ablation_full"),
        "--tag", f"ddim_s{N}",
        "--no-lpips",
    ]
    print("$ " + " ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
# === Cell 8: ARR grid on the +illum model at 5 steps ===
# Inference-time secondary contribution; cheap, 5 minutes total.
ARR_GRID = []
for ALPHA in (0.0, 0.1, 0.2, 0.3, 0.4, 0.5):
    tag = f"arr_a{int(ALPHA*100):03d}"
    cmd = [
        sys.executable, "evaluation.py",
        "--splits", f"lolv2_real:{LOLV2_LOW}:{LOLV2_HIGH}",
        "--inference-steps", "5",
        "--sampler", "ddim",
        "--checkpoint", ILLUM_CKPT,
        "--results-root", os.path.join(EVAL_RESULTS, "arr_grid"),
        "--tag", tag,
        "--gate-alpha", str(ALPHA),
        "--gate-floor", "0.5",
        "--no-lpips",
    ]
    print("$ " + " ".join(cmd))
    subprocess.run(cmd, check=True)
    summary_path = os.path.join(EVAL_RESULTS, f"arr_grid_{tag}", "summary.csv")
    with open(summary_path) as f:
        for row in csv.DictReader(f):
            row["alpha"] = ALPHA
            ARR_GRID.append(row)

print("\n### ARR grid (+illum model, LOL-v2 Real, 5 DDIM steps)\n")
print("| alpha | PSNR | SSIM |")
print("|---|---|---|")
for r in ARR_GRID:
    print(f"| {float(r['alpha']):.2f} | {float(r['psnr_mean']):.3f} | {float(r['ssim_mean']):.4f} |")

best_arr = max(ARR_GRID, key=lambda r: float(r['psnr_mean']))
print(f"\nBest ARR alpha: {float(best_arr['alpha'])}  ({float(best_arr['psnr_mean']):.3f} dB)")

In [ ]:
# === Cell 9: T4 efficiency for the +illum model ===
cmd = [
    sys.executable, "measure_efficiency.py",
    "--steps", "5", "10", "20",
    "--resolution", "400", "600",
    "--device", "cuda",
    "--checkpoint", ILLUM_CKPT,
    "--out-csv", os.path.join(EVAL_RESULTS, "efficiency_t4_illum.csv"),
]
print("$ " + " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# === Cell 10: render Figures 1 (teaser) and 3 (step ablation) for the +illum model ===
subprocess.run([
    sys.executable, "make_teaser_figure.py",
    "--pred-s5",  os.path.join(EVAL_RESULTS, "headline_illum_s5/lolv2_real"),
    "--pred-s20", os.path.join(EVAL_RESULTS, "headline_illum_s20/lolv2_real"),
    "--low",      LOLV2_LOW,
    "--gt",       LOLV2_HIGH,
    "--per-image-csv",     os.path.join(EVAL_RESULTS, "headline_illum_s5/per_image.csv"),
    "--per-image-csv-s20", os.path.join(EVAL_RESULTS, "headline_illum_s20/per_image.csv"),
    "--split", "lolv2_real",
    "--out",     os.path.join(EVAL_RESULTS, "figure1_teaser_illum.pdf"),
    "--out-png", os.path.join(EVAL_RESULTS, "figure1_teaser_illum.png"),
], check=True)

subprocess.run([
    sys.executable, "make_step_ablation_figure.py",
    "--eval-root", EVAL_RESULTS,
    "--out",     os.path.join(EVAL_RESULTS, "figure3_step_ablation_illum.pdf"),
    "--out-png", os.path.join(EVAL_RESULTS, "figure3_step_ablation_illum.png"),
], check=True)

In [ ]:
# === Cell 11: print the four tables that go into the paper ===
import csv

def read_summary(path):
    if not os.path.exists(path): return []
    return list(csv.DictReader(open(path)))

def fmt_row(r, variant):
    lp = r.get('lpips_mean')
    lp_s = f"{float(lp):.4f}" if lp not in (None, '', 'None') else '-'
    return (f"| {variant} | {r['split']} | {r['n']} | {r['inference_steps']} | "
            f"{float(r['psnr_mean']):.3f} | {float(r['ssim_mean']):.4f} | {lp_s} | "
            f"{float(r['runtime_mean']):.3f} |")

# Table A: headline (+illum at 5 and 20)
print("\n### Table A — Headline (+illum, LOL-v2 Real trained)\n")
print("| Variant | Split | n | Steps | PSNR | SSIM | LPIPS | Latency/img (s) |")
print("|---|---|---|---|---|---|---|---|")
for tag in ("s5", "s20"):
    for r in read_summary(os.path.join(EVAL_RESULTS, f"headline_illum_{tag}/summary.csv")):
        print(fmt_row(r, "+illum"))

# Table B: method ablation (baseline vs +illum at 5 steps)
print("\n### Table B — Method ablation (5 DDIM steps, both LOL-v2 Real trained)\n")
print("| Variant | Split | n | Steps | PSNR | SSIM | LPIPS | Latency/img (s) |")
print("|---|---|---|---|---|---|---|---|")
for r in read_summary(os.path.join(EVAL_RESULTS, "method_ablation_baseline_s5/summary.csv")):
    print(fmt_row(r, "baseline (no prior)"))
for r in read_summary(os.path.join(EVAL_RESULTS, "headline_illum_s5/summary.csv")):
    print(fmt_row(r, "+ illum prior (ours)"))

# Table C: step ablation
print("\n### Table C — Step ablation (+illum)\n")
print("| Steps | Split | PSNR | SSIM | Latency/img (s) |")
print("|---|---|---|---|---|")
for N in (5, 10, 20, 50, 100):
    for r in read_summary(os.path.join(EVAL_RESULTS, f"step_ablation_full_ddim_s{N}/summary.csv")):
        print(f"| {N} | {r['split']} | {float(r['psnr_mean']):.3f} | {float(r['ssim_mean']):.4f} | {float(r['runtime_mean']):.3f} |")

# Table D: ARR
print("\n### Table D — Adaptive Residual Rescaling (+illum, 5 DDIM steps, LOL-v2 Real)\n")
print("| alpha | PSNR | SSIM |")
print("|---|---|---|")
for r in ARR_GRID:
    print(f"| {float(r['alpha']):.2f} | {float(r['psnr_mean']):.3f} | {float(r['ssim_mean']):.4f} |")

# Efficiency
print("\n### Efficiency (T4, 400x600, +illum)\n")
with open(os.path.join(EVAL_RESULTS, "efficiency_t4_illum.csv")) as f:
    print(f.read())

In [ ]:
# === Cell 12: bundle everything for download ===
import shutil
out_zip = "/kaggle/working/phase3_day2_eval_outputs.zip"
if os.path.exists(out_zip): os.remove(out_zip)
shutil.make_archive(out_zip.replace(".zip", ""), "zip", EVAL_RESULTS)
print("Wrote", out_zip)
print("Download via Kaggle right panel -> Output tab.")